# 09 — Export pour GAMA

Objectif : préparer les fichiers que GAMA va importer :
- `hanoi_roads.shp` — réseau routier (agents véhicules)
- `hanoi_buildings.shp` — bâtiments (obstacles à la propagation)
- `hanoi_noise_map.csv` — niveaux de bruit prédits point par point

Dans GAMA, tu importeras ces fichiers comme `shape_file` et tu lanceras
la simulation de propagation par-dessus.

In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import os

STUDY_AREA = 'Bach Khoa, Hanoi, Vietnam'
OUT_DIR    = '../outputs/maps/gama_inputs/'
os.makedirs(OUT_DIR, exist_ok=True)

/Users/phocidae/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# 1. Réseau routier → shapefile
G = ox.load_graphml('../data/processed/hanoi_roads.graphml')
_, edges = ox.graph_to_gdfs(G)

# Garde seulement les colonnes utiles pour GAMA
road_cols = ['geometry', 'highway', 'name', 'length', 'oneway']
road_cols = [c for c in road_cols if c in edges.columns]
edges[road_cols].to_file(OUT_DIR + 'roads.shp')
print('roads.shp exporté')

roads.shp exporté


In [3]:
# 2. Bâtiments → shapefile
buildings = ox.features_from_place(STUDY_AREA, tags={'building': True})
buildings = buildings[buildings.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
buildings[['geometry']].to_file(OUT_DIR + 'buildings.shp')
print(f'{len(buildings)} bâtiments exportés → buildings.shp')

106 bâtiments exportés → buildings.shp


In [4]:
# 3. Carte de bruit → CSV formaté pour GAMA
noise = pd.read_csv('../outputs/maps/hanoi_noise_map.csv')

# GAMA attend : x (longitude), y (latitude), noise_dB
gama_noise = noise.rename(columns={
    'longitude': 'x',
    'latitude':  'y',
    'noise_pred_dB': 'noise_dB'
})[['x', 'y', 'noise_dB']]

gama_noise.to_csv(OUT_DIR + 'noise_map.csv', index=False)
print(f'{len(gama_noise)} points exportés → noise_map.csv')

print('\nFichiers prêts pour GAMA :')
for f in os.listdir(OUT_DIR):
    print(f'  {OUT_DIR}{f}')

8640 points exportés → noise_map.csv

Fichiers prêts pour GAMA :
  ../outputs/maps/gama_inputs/buildings.dbf
  ../outputs/maps/gama_inputs/noise_map.csv
  ../outputs/maps/gama_inputs/buildings.shx
  ../outputs/maps/gama_inputs/buildings.shp
  ../outputs/maps/gama_inputs/buildings.cpg
  ../outputs/maps/gama_inputs/roads.prj
  ../outputs/maps/gama_inputs/roads.dbf
  ../outputs/maps/gama_inputs/roads.shx
  ../outputs/maps/gama_inputs/buildings.prj
  ../outputs/maps/gama_inputs/roads.cpg
  ../outputs/maps/gama_inputs/roads.shp


## Prochaine étape : GAMA

Dans ton modèle GAMA (fichier `.gaml`) :

```gaml
file roads_file    <- file('../outputs/maps/gama_inputs/roads.shp');
file buildings_file <- file('../outputs/maps/gama_inputs/buildings.shp');
file noise_file    <- csv_file('../outputs/maps/gama_inputs/noise_map.csv', true);
```

Les agents `vehicle` se déplacent sur `roads`, les agents `building` bloquent
la propagation, et `noise_map` initialise les niveaux de base.